In [6]:
# ============================================================
# PMSM ELECTRO-THERMAL PHYSICS DEFENSIBILITY ANALYSIS
# ============================================================
#
# PURPOSE
# -------
# This script evaluates whether the proposed electro-thermal
# physics formulation is numerically and physically defensible
# on the raw PMSM dataset.
#
# IMPORTANT:
#   1. We DO NOT assume derivative columns already exist.
#   2. Derivatives are calculated within each profile.
#   3. Profiles are never differentiated across boundaries.
#   4. We use the actual columns present in the dataset.
#
# RAW COLUMNS EXPECTED:
# u_q, coolant, stator_winding, u_d, stator_tooth,
# motor_speed, i_d, i_q, pm, stator_yoke, ambient,
# torque, profile_id
#
# ============================================================

import numpy as np
import pandas as pd

print("=" * 80)
print("PMSM ELECTRO-THERMAL PHYSICS DEFENSIBILITY ANALYSIS")
print("=" * 80)


# ============================================================
# 0. DATAFRAME CHECK
# ============================================================

# IMPORTANT:
# Your dataframe must be called df.
#
# If your dataframe has another name, change it here.
#
# Example:
df = pd.read_csv("measures_v2.csv")

if not isinstance(df, pd.DataFrame):
    raise TypeError(
        "The variable 'df' is not a pandas DataFrame.\n"
        "Load your dataset into a DataFrame named 'df'."
    )

print("\nDataset shape:")
print(df.shape)

print("\nColumns:")
print(df.columns.tolist())


# ============================================================
# 1. COLUMN DEFINITIONS
# ============================================================

PROFILE = "profile_id"

UD = "u_d"
UQ = "u_q"

ID = "i_d"
IQ = "i_q"

SPEED = "motor_speed"

TORQUE = "torque"

TW = "stator_winding"
TT = "stator_tooth"
TY = "stator_yoke"
TPM = "pm"

COOLANT = "coolant"
AMBIENT = "ambient"


REQUIRED_COLUMNS = [
    PROFILE,
    UD,
    UQ,
    ID,
    IQ,
    SPEED,
    TORQUE,
    TW,
    TT,
    TY,
    TPM,
    COOLANT,
    AMBIENT
]


# ============================================================
# 2. VALIDATE COLUMNS
# ============================================================

missing = [
    c for c in REQUIRED_COLUMNS
    if c not in df.columns
]

if missing:

    print("\nERROR: Missing columns:")
    for c in missing:
        print("  -", c)

    raise KeyError(
        "\nThe dataset does not contain all required columns."
    )

print("\nAll required columns are present.")


# ============================================================
# 3. COPY DATA
# ============================================================

data = df[
    REQUIRED_COLUMNS
].copy()


# ============================================================
# 4. NUMERIC CONVERSION
# ============================================================

for col in REQUIRED_COLUMNS:

    if col != PROFILE:

        data[col] = pd.to_numeric(
            data[col],
            errors="coerce"
        )


# Remove invalid rows
before = len(data)

data = data.dropna(
    subset=REQUIRED_COLUMNS
).copy()

after = len(data)

print("\nRows removed because of NaN/non-numeric values:",
      before - after)


# ============================================================
# 5. SORT BY PROFILE
# ============================================================
#
# IMPORTANT:
# We must calculate temporal derivatives separately for each
# operating profile.
#
# We NEVER calculate a derivative across profile boundaries.
#
# ============================================================

data = data.sort_values(
    by=[PROFILE]
).reset_index(drop=True)


print("\nNumber of profiles:",
      data[PROFILE].nunique())


# ============================================================
# 6. BASIC PHYSICAL QUANTITIES
# ============================================================

# ------------------------------------------------------------
# Electrical apparent magnitude
# ------------------------------------------------------------

data["voltage_mag"] = np.sqrt(
    data[UD] ** 2 +
    data[UQ] ** 2
)


data["current_mag"] = np.sqrt(
    data[ID] ** 2 +
    data[IQ] ** 2
)


# ------------------------------------------------------------
# dq electrical power
#
# IMPORTANT:
#
# Depending on the Park transformation convention,
# the scaling factor may be 3/2.
#
# We therefore explicitly define it.
#
# P_e = 3/2 (u_d i_d + u_q i_q)
#
# ------------------------------------------------------------

POWER_SCALE = 1.5

data["P_electrical"] = (
    POWER_SCALE *
    (
        data[UD] * data[ID] +
        data[UQ] * data[IQ]
    )
)


# ============================================================
# 7. COPPER LOSS
# ============================================================
#
# For a three-phase PMSM:
#
# P_cu = 3 R_s (i_d^2 + i_q^2)
#
# However, the resistance must be known.
#
# We DO NOT claim a numerical copper loss is physically
# exact without a validated R_s.
#
# Therefore Rs is explicitly declared below.
#
# ============================================================

RS = 0.05   # Ohm
# IMPORTANT:
# Replace with the validated stator resistance if available.

data["P_copper"] = (
    3.0 *
    RS *
    (
        data[ID] ** 2 +
        data[IQ] ** 2
    )
)


# ============================================================
# 8. THERMAL TEMPERATURE DIFFERENCES
# ============================================================

data["dTw"] = (
    data[TW] -
    data[COOLANT]
)

data["dTt_y"] = (
    data[TT] -
    data[TY]
)

data["dTt_pm"] = (
    data[TT] -
    data[TPM]
)

data["dTy_c"] = (
    data[TY] -
    data[COOLANT]
)

data["dTy_a"] = (
    data[TY] -
    data[AMBIENT]
)


# ============================================================
# 9. TEMPORAL DERIVATIVES
# ============================================================
#
# Your previous code failed because it expected:
#
# dTw_dt
# dTT_dt
# dTY_dt
# dTPM_dt
#
# These columns do not exist in the raw dataset.
#
# We calculate them here.
#
# ------------------------------------------------------------
# IMPORTANT:
#
# The dataset excerpt does not contain an explicit time column.
#
# Therefore:
#
#       dt = 1
#
# means "one dataset sample interval".
#
# This means derivatives are currently in:
#
#       temperature / sample
#
# rather than:
#
#       temperature / second
#
# For the final paper, replace SAMPLE_TIME with the actual
# sampling interval if known.
#
# ============================================================

SAMPLE_TIME = 1.0


def calculate_profile_derivative(
    dataframe,
    column,
    profile_column=PROFILE,
    dt=SAMPLE_TIME
):

    result = pd.Series(
        index=dataframe.index,
        dtype=np.float64
    )

    for profile_id, group in dataframe.groupby(
        profile_column,
        sort=False
    ):

        values = group[column].to_numpy(
            dtype=np.float64
        )

        if len(values) < 2:

            derivative = np.zeros(
                len(values),
                dtype=np.float64
            )

        else:

            derivative = np.gradient(
                values,
                dt
            )

        result.loc[group.index] = derivative

    return result


# ------------------------------------------------------------
# Calculate thermal derivatives
# ------------------------------------------------------------

data["dTw_dt"] = calculate_profile_derivative(
    data,
    TW
)

data["dTT_dt"] = calculate_profile_derivative(
    data,
    TT
)

data["dTY_dt"] = calculate_profile_derivative(
    data,
    TY
)

data["dTPM_dt"] = calculate_profile_derivative(
    data,
    TPM
)


print("\nDerivative columns successfully created.")


# ============================================================
# 10. ELECTRICAL STATE DERIVATIVES
# ============================================================

data["did_dt"] = calculate_profile_derivative(
    data,
    ID
)

data["diq_dt"] = calculate_profile_derivative(
    data,
    IQ
)


# ============================================================
# 11. BASIC STATISTICS
# ============================================================

print("\n")
print("=" * 80)
print("BASIC PHYSICAL STATISTICS")
print("=" * 80)

physical_columns = [
    UD,
    UQ,
    ID,
    IQ,
    SPEED,
    TORQUE,
    TW,
    TT,
    TY,
    TPM,
    COOLANT,
    AMBIENT,
    "P_copper",
    "P_electrical",
    "dTw_dt",
    "dTT_dt",
    "dTY_dt",
    "dTPM_dt"
]

print(
    data[physical_columns].describe().T[
        ["mean", "std", "min", "max"]
    ]
)


# ============================================================
# 12. THERMAL CORRELATION ANALYSIS
# ============================================================

thermal_columns = [
    TW,
    TT,
    TY,
    TPM,
    COOLANT,
    AMBIENT,
    "dTw",
    "dTt_y",
    "dTt_pm",
    "dTy_c",
    "dTy_a",
    "P_copper"
]

thermal_corr = data[
    thermal_columns
].corr()


print("\n")
print("=" * 80)
print("THERMAL CORRELATION MATRIX")
print("=" * 80)

print(
    thermal_corr.round(4)
)


# ============================================================
# 13. ELECTRO-THERMAL CORRELATION
# ============================================================

electro_thermal_columns = [
    ID,
    IQ,
    UD,
    UQ,
    SPEED,
    TORQUE,
    "P_copper",
    TW,
    TT,
    TY,
    TPM,
    COOLANT,
    AMBIENT
]

electro_thermal_corr = data[
    electro_thermal_columns
].corr()


print("\n")
print("=" * 80)
print("ELECTRO-THERMAL CORRELATION MATRIX")
print("=" * 80)

print(
    electro_thermal_corr.round(4)
)


# ============================================================
# 14. PROFILE-LEVEL THERMAL RANGE
# ============================================================

profile_range = (
    data
    .groupby(PROFILE)
    .agg(
        Tw_range=(TW, lambda x: x.max() - x.min()),
        Tt_range=(TT, lambda x: x.max() - x.min()),
        Ty_range=(TY, lambda x: x.max() - x.min()),
        Tpm_range=(TPM, lambda x: x.max() - x.min()),
        Pcu_range=(
            "P_copper",
            lambda x: x.max() - x.min()
        )
    )
    .reset_index()
)


print("\n")
print("=" * 80)
print("PROFILE-LEVEL THERMAL / COPPER-LOSS RANGES")
print("=" * 80)

print(
    profile_range.to_string(
        index=False
    )
)


# ============================================================
# 15. THERMAL DYNAMICS CORRELATION
# ============================================================

thermal_dynamic_columns = [
    "dTw_dt",
    "dTT_dt",
    "dTY_dt",
    "dTPM_dt",
    "P_copper"
]

thermal_dynamic_corr = data[
    thermal_dynamic_columns
].corr()


print("\n")
print("=" * 80)
print("THERMAL DYNAMIC CORRELATION")
print("=" * 80)

print(
    thermal_dynamic_corr.round(4)
)


# ============================================================
# 16. CHECK THERMAL RESPONSE TO COPPER LOSS
# ============================================================
#
# A physically plausible thermal model should generally show
# that copper-loss variation has a meaningful relationship
# with temperature dynamics.
#
# We inspect:
#
# P_copper <-> dTw/dt
# P_copper <-> dTT/dt
# P_copper <-> dTY/dt
# P_copper <-> dTPM/dt
#
# ============================================================

print("\n")
print("=" * 80)
print("COPPER LOSS → THERMAL DYNAMICS")
print("=" * 80)

for col in [
    "dTw_dt",
    "dTT_dt",
    "dTY_dt",
    "dTPM_dt"
]:

    corr = data[
        "P_copper"
    ].corr(
        data[col]
    )

    print(
        f"P_copper vs {col:10s} : {corr: .6f}"
    )


# ============================================================
# 17. ELECTRICAL POWER CONSISTENCY
# ============================================================
#
# Mechanical output power:
#
# P_mech = T * omega
#
# omega = 2*pi*n / 60
#
# IMPORTANT:
#
# motor_speed must be interpreted correctly.
#
# If motor_speed is RPM:
#
#       omega = 2*pi*n/60
#
# If it is rad/s, this must NOT be used.
#
# The dataset values you showed are very small, so we explicitly
# calculate the RPM interpretation here for diagnostic purposes.
#
# ============================================================

data["omega_rpm_assumption"] = (
    2.0 *
    np.pi *
    data[SPEED] /
    60.0
)

data["P_mechanical_rpm_assumption"] = (
    data[TORQUE] *
    data["omega_rpm_assumption"]
)


print("\n")
print("=" * 80)
print("ELECTRICAL / MECHANICAL POWER DIAGNOSTIC")
print("=" * 80)

power_columns = [
    "P_electrical",
    "P_mechanical_rpm_assumption",
    "P_copper"
]

print(
    data[power_columns].describe().T[
        ["mean", "std", "min", "max"]
    ]
)


# ============================================================
# 18. POWER BALANCE DIAGNOSTIC
# ============================================================
#
# A simplified energy balance is:
#
# P_electrical ≈ P_mechanical + P_losses
#
# We cannot assert that P_losses = P_copper only because
# additional losses may exist:
#
#   - iron losses
#   - mechanical losses
#   - inverter losses
#   - stray losses
#
# Therefore this is a diagnostic, NOT yet the final physics
# equation.
#
# ============================================================

data["power_balance_residual_simple"] = (
    data["P_electrical"]
    -
    data["P_mechanical_rpm_assumption"]
    -
    data["P_copper"]
)


print("\n")
print("=" * 80)
print("SIMPLE POWER BALANCE RESIDUAL")
print("=" * 80)

print(
    data[
        "power_balance_residual_simple"
    ].describe()[
        ["mean", "std", "min", "max"]
    ]
)


# ============================================================
# 19. NORMALIZED POWER BALANCE ERROR
# ============================================================

denominator = (
    np.abs(data["P_electrical"])
    + 1e-8
)

data["relative_power_balance_error"] = (
    np.abs(
        data["power_balance_residual_simple"]
    )
    /
    denominator
)


print("\nMean relative power-balance error:")
print(
    data[
        "relative_power_balance_error"
    ].mean()
)


print("\nMedian relative power-balance error:")
print(
    data[
        "relative_power_balance_error"
    ].median()
)


# ============================================================
# 20. TEMPERATURE DIFFERENCE STATISTICS
# ============================================================

delta_columns = [
    "dTw",
    "dTt_y",
    "dTt_pm",
    "dTy_c",
    "dTy_a"
]


print("\n")
print("=" * 80)
print("THERMAL TEMPERATURE DIFFERENCES")
print("=" * 80)

print(
    data[delta_columns].describe().T[
        ["mean", "std", "min", "max"]
    ]
)


# ============================================================
# 21. THERMAL ORDERING CHECK
# ============================================================
#
# We examine how often the following physically plausible
# relationships occur:
#
# Stator winding > coolant
# Stator tooth > coolant
# Stator yoke > coolant
#
# These are NOT universal laws; they are empirical diagnostics.
#
# ============================================================

data["Tw_gt_coolant"] = (
    data[TW] >
    data[COOLANT]
)

data["Tt_gt_coolant"] = (
    data[TT] >
    data[COOLANT]
)

data["Ty_gt_coolant"] = (
    data[TY] >
    data[COOLANT]
)


print("\n")
print("=" * 80)
print("THERMAL ORDERING DIAGNOSTIC")
print("=" * 80)

print(
    "Tw > coolant :",
    data["Tw_gt_coolant"].mean()
)

print(
    "Tt > coolant :",
    data["Tt_gt_coolant"].mean()
)

print(
    "Ty > coolant :",
    data["Ty_gt_coolant"].mean()
)


# ============================================================
# 22. ELECTRICAL STATE DYNAMICS
# ============================================================

electrical_dynamic_columns = [
    "did_dt",
    "diq_dt",
    ID,
    IQ,
    UD,
    UQ,
    SPEED
]


print("\n")
print("=" * 80)
print("ELECTRICAL DYNAMICS")
print("=" * 80)

print(
    data[electrical_dynamic_columns]
    .describe()
    .T[
        ["mean", "std", "min", "max"]
    ]
)


# ============================================================
# 23. ELECTRICAL CORRELATIONS
# ============================================================

electrical_columns = [
    UD,
    UQ,
    ID,
    IQ,
    SPEED,
    "did_dt",
    "diq_dt",
    TORQUE
]

electrical_corr = data[
    electrical_columns
].corr()


print("\n")
print("=" * 80)
print("ELECTRICAL CORRELATION MATRIX")
print("=" * 80)

print(
    electrical_corr.round(4)
)


# ============================================================
# 24. DATASET-LEVEL PHYSICS SUMMARY
# ============================================================

summary = {

    "n_records":
        len(data),

    "n_profiles":
        data[PROFILE].nunique(),

    "temperature_range_winding":
        data[TW].max() - data[TW].min(),

    "temperature_range_tooth":
        data[TT].max() - data[TT].min(),

    "temperature_range_yoke":
        data[TY].max() - data[TY].min(),

    "temperature_range_pm":
        data[TPM].max() - data[TPM].min(),

    "copper_loss_range":
        data["P_copper"].max()
        -
        data["P_copper"].min(),

    "corr_Pcu_dTw_dt":
        data[
            "P_copper"
        ].corr(
            data["dTw_dt"]
        ),

    "corr_Pcu_dTT_dt":
        data[
            "P_copper"
        ].corr(
            data["dTT_dt"]
        ),

    "corr_Pcu_dTY_dt":
        data[
            "P_copper"
        ].corr(
            data["dTY_dt"]
        ),

    "corr_Pcu_dTPM_dt":
        data[
            "P_copper"
        ].corr(
            data["dTPM_dt"]
        ),

    "mean_relative_power_balance_error":
        data[
            "relative_power_balance_error"
        ].mean(),

    "median_relative_power_balance_error":
        data[
            "relative_power_balance_error"
        ].median()
}


print("\n")
print("=" * 80)
print("FINAL PHYSICS DATASET SUMMARY")
print("=" * 80)

for key, value in summary.items():

    print(
        f"{key:45s}: {value}"
    )


# ============================================================
# 25. SAVE DERIVED DATA
# ============================================================
#
# This gives us a reproducible physics-analysis dataset.
#
# ============================================================

OUTPUT_FILE = "pmsm_physics_defensibility_data.csv"

data.to_csv(
    OUTPUT_FILE,
    index=False
)

print("\n")
print("=" * 80)
print("ANALYSIS COMPLETE")
print("=" * 80)

print(
    f"Derived dataset saved to: {OUTPUT_FILE}"
)

print(
    "\nIMPORTANT:"
)

print(
    "Do NOT use the resulting correlations as proof that "
    "the governing equations are correct."
)

print(
    "They are diagnostics used to determine whether the "
    "dataset supports the proposed physical formulation."
)

PMSM ELECTRO-THERMAL PHYSICS DEFENSIBILITY ANALYSIS

Dataset shape:
(1330816, 13)

Columns:
['u_q', 'coolant', 'stator_winding', 'u_d', 'stator_tooth', 'motor_speed', 'i_d', 'i_q', 'pm', 'stator_yoke', 'ambient', 'torque', 'profile_id']

All required columns are present.

Rows removed because of NaN/non-numeric values: 0

Number of profiles: 69

Derivative columns successfully created.


BASIC PHYSICAL STATISTICS
                       mean           std           min           max
u_d              -25.133809     63.091972 -1.315304e+02    131.469788
u_q               54.279005     44.173234 -2.529093e+01    133.036994
i_d              -68.716810     64.933233 -2.780036e+02      0.051897
i_q               37.412782     92.181880 -2.934268e+02    301.707855
motor_speed     2202.080728   1859.663350 -2.755491e+02   6000.015137
torque            31.106032     77.135755 -2.464667e+02    261.005707
stator_winding    66.342745     28.672061  1.858582e+01    141.362885
stator_tooth      56.87